<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network 徽标">
    </a>
</p>


<h1>实验：使用 Softmax 进行手写数字图像分类</h1>


<p>预计所需时间：<strong>25 分钟</strong></p>


## 概述
在本实验中，你将使用单层 Softmax 分类器对 MNIST 数据库中的手写数字进行分类。


<h2>目标</h2>

<ul>
    <li>下载 MNIST 训练和验证数字图片</li>
    <li>使用 PyTorch 创建 Softmax 分类器</li>
    <li>创建准则、优化器和数据加载器</li>
    <li>创建数据加载器并设置批量大小</li>
    <li>训练模型</li>
    <li>分析结果和模型</li>
</ul> 


<h2>目录</h2>
本笔记本按以下章节组织：

-   [准备数据](#Make-Some-Data)
-   [构建 Softmax 分类器](#Build-a-Softmax-Classifer)
-   [定义 Softmax 分类器、准则函数、优化器并训练模型](#Define-the-Softmax-Classifier,-Criterion-Function,-Optimizer,-and-Train-the-Model)
-   [分析结果](#Analyze-Results)
<hr>


<h2>准备</h2>


我们需要以下库


In [ ]:
%%time
%pip install numpy matplotlib
%pip install torch==2.8.0+cpu torchvision==0.23.0+cpu torchaudio==2.8.0+cpu \
    --index-url https://download.pytorch.org/whl/cpu

In [ ]:
# 导入本实验所需的库

# 使用以下代码行安装 torchvision 库
# 使用以下命令安装 torchvision 库

# PyTorch 库
import torch 
# PyTorch 神经网络模块
import torch.nn as nn
# 转换数据
import torchvision.transforms as transforms
# 获取手写数字数据集
import torchvision.datasets as dsets
# 绘制图像
import matplotlib.pylab as plt
# 使用数组来操作和存储数据
import numpy as np

使用以下函数绘制 Softmax 函数的参数：


In [ ]:
# 绘制参数的函数

def PlotParameters(model): 
    W = model.state_dict()['linear.weight'].data
    w_min = W.min().item()
    w_max = W.max().item()
    fig, axes = plt.subplots(2, 5)
    fig.subplots_adjust(hspace=0.01, wspace=0.1)
    for i, ax in enumerate(axes.flat):
        if i < 10:
            
            # 设置子图的标签。
            ax.set_xlabel("class: {0}".format(i))

            # 绘制图片。
            ax.imshow(W[i, :].view(28, 28), vmin=w_min, vmax=w_max, cmap='seismic')

            ax.set_xticks([])
            ax.set_yticks([])

        # 确保多个图像在同一个 Notebook 单元格中正确显示
        # 在单个 Notebook 单元格中。
    plt.show()

使用以下函数可视化数据： 


In [ ]:
# 绘制数据
def show_data(data_sample):
    plt.imshow(data_sample[0].numpy().reshape(28, 28), cmap='gray')
    plt.title('y = ' + str(data_sample[1]))


<!--Empty Space for separating topics-->


<h2 id="Makeup_Data">准备数据</h2> 


通过将 <code>train</code> 参数设置为 <code>True</code> 来加载<em>训练</em>数据集，并通过在 <code>transform</code> 参数中放置一个转换对象将其转换为张量。


In [ ]:
# 创建并打印训练数据集

train_dataset = dsets.MNIST(root='./data', train=True, download=True, transform=transforms.ToTensor())
print("Print the training dataset:\n ", train_dataset)

加载<em>测试</em>数据集，并通过在 <code>transform</code> 参数中放置一个转换对象将其转换为张量。


In [ ]:
# 创建并打印验证数据集

validation_dataset = dsets.MNIST(root='./data', download=True, transform=transforms.ToTensor())
print("Print the validation dataset:\n ", validation_dataset)

我们可以通过索引 train_dataset 和 test_dataset 来访问数据


In [ ]:
# 打印第一张图片和标签

print("First Image and Label") 
show_data(train_dataset[0])

矩形张量中的每个元素对应一个代表像素强度的数值，如下图所示：


<img src="https://s3-api.us-geo.objectstorage.softlayer.net/cf-courses-data/CognitiveClass/DL0110EN/notebook_images%20/chapter3/3.32_image_values.png" width="550" alt="MNIST 元素">


在这张图片中，数值是反的，即黑色代表白色。


打印第四个元素的标签：


In [ ]:
# 打印标签

print("The label: ", train_dataset[3][1])

结果显示图片中的数字是 1


绘制第四个样本：


In [ ]:
# 绘制图片

print("The image: ")
show_data(train_dataset[3])

你可以看到它是 1。现在，绘制第三个样本：


In [ ]:
# 绘制图片

show_data(train_dataset[2])

<!--Empty Space for separating topics-->


<h2 id="#Classifier">构建 Softmax 分类器</h2>


构建一个 Softmax 分类器类： 


In [ ]:
# 定义 softmax 分类器类
# 继承 nn.Module，它是所有神经网络的基类
class SoftMax(nn.Module):
    
    # 构造函数
    def __init__(self, input_size, output_size):
        super(SoftMax, self).__init__()
        # 创建给定输入大小和输出大小的层
        self.linear = nn.Linear(input_size, output_size)
        
    # 预测
    def forward(self, x):
        # 将 x 值传入上面定义的单个层
        z = self.linear(x)
        return z

Softmax 函数需要向量输入。注意向量的形状是 28×28。


In [ ]:
# 打印训练数据集的形状

train_dataset[0][0].shape

按照下图所示展平张量： 


<img src="https://s3-api.us-geo.objectstorage.softlayer.net/cf-courses-data/CognitiveClass/DL0110EN/notebook_images%20/chapter3/3.3.2image_to_vector.gif" width="550" alt="展平图片">


张量的大小现在是 784。


<img src="https://s3-api.us-geo.objectstorage.softlayer.net/cf-courses-data/CognitiveClass/DL0110EN/notebook_images%20/chapter3/3.3.2Imagetovector2.png" width="550" alt="展平图片">


设置输入大小和输出大小： 


In [ ]:
# 设置输入大小和输出大小

input_dim = 28 * 28
output_dim = 10

<!--Empty Space for separating topics-->


<h2 id="Model">定义 Softmax 分类器、准则函数、优化器并训练模型</h2> 


In [ ]:
# 创建模型
# 输入维度是 28*28，即图片转换成的张量
# 输出维度为 10，因为图片可能是 10 个数字之一
model = SoftMax(input_dim, output_dim)
print("Print the model:\n ", model)

查看模型参数的大小： 


In [ ]:
# 打印参数

print('W: ',list(model.parameters())[0].size())
print('b: ',list(model.parameters())[1].size())

你可以将每个类别的模型参数转换为矩形网格：  


<a>     <img src="https://s3-api.us-geo.objectstorage.softlayer.net/cf-courses-data/CognitiveClass/DL0110EN/notebook_images%20/chapter3/3.3.2paramaters_to_image.gif" width="550," align="center"></a> 


将每个类别的模型参数绘制成方形图像： 


In [ ]:
# 绘制每个类别的模型参数
# 由于模型尚未训练，参数看起来是随机的

PlotParameters(model)

我们可以进行预测


In [ ]:
# 首先获取第一张图片的 X 值
X = train_dataset[0][0]
# 可以看到形状是 1×28×28，我们需要将其展平为 1×(28×28)（即 784）
print(X.shape)
X = X.view(-1, 28*28)
print(X.shape)
# 现在我们可以进行预测，每个类别都有一个值，值越高表示模型越确信是该数字
model(X)

定义学习率、优化器、准则、数据加载器：


In [ ]:
# 定义学习率、优化器、准则和数据加载器

learning_rate = 0.1
# 优化器将使用学习率更新模型参数
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
# 准则将度量预测值与真实标签值之间的损失
# 这里发生 SoftMax，它被内置于交叉熵损失准则中
criterion = nn.CrossEntropyLoss()
# 创建训练数据加载器以便设置批量大小
train_loader = torch.utils.data.DataLoader(dataset=train_dataset, batch_size=100)
# 创建验证数据加载器以便设置批量大小
validation_loader = torch.utils.data.DataLoader(dataset=validation_dataset, batch_size=5000)

### 交叉熵损失如何使用 SoftMax


我们有 X（第一张图片的 X 值）和 `actual`（图片所属的数字类别）。输出 `model_output` 是模型为该图片每个类别分配的值。


In [ ]:
model_output = model(X)
actual = torch.tensor([train_dataset[0][1]])

show_data(train_dataset[0])
print("Output: ", model_output)
print("Actual:", actual)

准则将接收这些值并返回损失


In [ ]:
criterion(model_output, actual)

交叉熵损失接收概率，而我们可以看到 `model_output` 并不是概率，这正是 softmax 发挥作用的地方


In [ ]:
softmax = nn.Softmax(dim=1)
probability = softmax(model_output)
print(probability)

现在我们有了概率，可以计算该图片所属类别的概率的负对数。图片属于目标类别，因此我们计算目标索引处概率的负对数。


In [ ]:
-1*torch.log(probability[0][actual])

如你所见，上面的结果与准则的结果一致，这就是交叉熵损失如何使用 Softmax。


### 训练


训练模型并确定验证准确率**（可能需要几分钟）**： 


In [ ]:
# 使用训练数据训练模型的次数
n_epochs = 10
# 用于记录损失和准确率的列表
loss_list = []
accuracy_list = []
# 验证数据的大小
N_test = len(validation_dataset)

# 根据 epoch 数量训练模型的函数
def train_model(n_epochs):
    # 循环 n_epochs 次
    for epoch in range(n_epochs):
        # 对于训练加载器中的每个批次
        for x, y in train_loader:
            # 重置计算得到的梯度值，每次都必须这样做，因为如果不重置就会累积
            optimizer.zero_grad()
            # 基于图片张量进行预测
            z = model(x.view(-1, 28 * 28))
            # 计算模型输出与真实类别之间的损失
            loss = criterion(z, y)
            # 计算每个权重和偏置的梯度值
            loss.backward()
            # 根据计算得到的梯度值更新权重和偏置
            optimizer.step()
        
        # 每个 epoch 我们检查模型在未见过数据（即验证数据）上的表现，这里不进行训练
        correct = 0
        # 对于验证加载器中的每个批次
        for x_test, y_test in validation_loader:
            # 基于图片张量进行预测
            z = model(x_test.view(-1, 28 * 28))
            # 找到输出最高的类别
            _, yhat = torch.max(z.data, 1)
            # 检查预测是否与真实类别匹配，如果匹配则增加正确计数
            correct += (yhat == y_test).sum().item()
        # 通过将正确预测数除以验证数据集大小来计算准确率
        accuracy = correct / N_test
        # 记录损失
        loss_list.append(loss.data)
        # 记录准确率
        accuracy_list.append(accuracy)

# 函数调用
train_model(n_epochs)

<!--Empty Space for separating topics-->


<h2 id="Result">分析结果</h2> 


绘制验证数据上的损失和准确率：


In [ ]:
# 绘制损失和准确率

fig, ax1 = plt.subplots()
color = 'tab:red'
ax1.plot(loss_list,color=color)
ax1.set_xlabel('epoch',color=color)
ax1.set_ylabel('total loss',color=color)
ax1.tick_params(axis='y', color=color)
    
ax2 = ax1.twinx()  
color = 'tab:blue'
ax2.set_ylabel('accuracy', color=color)  
ax2.plot( accuracy_list, color=color)
ax2.tick_params(axis='y', color=color)
fig.tight_layout()

查看训练后每个类别的参数结果。你可以看到它们看起来像对应的数字。 


In [ ]:
# 绘制参数

PlotParameters(model)

我们绘制前五个分类错误的样本以及该类别的概率。


In [ ]:
# 绘制分类错误的样本
Softmax_fn=nn.Softmax(dim=-1)
count = 0
for x, y in validation_dataset:
    z = model(x.reshape(-1, 28 * 28))
    _, yhat = torch.max(z, 1)
    if yhat != y:
        show_data((x, y))
        plt.show()
        print("yhat:", yhat)
        print("probability of class ", torch.max(Softmax_fn(z)).item())
        count += 1
    if count >= 5:
        break       

<!--Empty Space for separating topics-->


我们绘制前五个正确分类的样本以及该类别的概率。我们可以看到概率要大得多。


In [ ]:
# 绘制分类正确的样本
Softmax_fn=nn.Softmax(dim=-1)
count = 0
for x, y in validation_dataset:
    z = model(x.reshape(-1, 28 * 28))
    _, yhat = torch.max(z, 1)
    if yhat == y:
        show_data((x, y))
        plt.show()
        print("yhat:", yhat)
        print("probability of class ", torch.max(Softmax_fn(z)).item())
        count += 1
    if count >= 5:
        break  

<h2>关于作者：</h2> 

<a href="https://www.linkedin.com/in/joseph-s-50398b136/">Joseph Santarcangelo</a> 拥有电气工程博士学位，其研究专注于使用机器学习、信号处理和计算机视觉来确定视频如何影响人类认知。Joseph 自完成博士学位以来一直在 IBM 工作。


其他贡献者：<a href="https://www.linkedin.com/in/michelleccarey/">Michelle Carey</a>、<a href="https://www.linkedin.com/in/jiahui-mavis-zhou-a4537814a">Mavis Zhou</a>


<!--## Change Log

| 日期（YYYY-MM-DD） | 版本 | 修改者 | 变更说明                                          |
| ----------------- | ------- | ---------- | ----------------------------------------------------------- |
| 2025-07-10        | 2.0     | Sathya    | 将实验转换为 Jupyter current 版本 |
| 2020-09-23        | 2.0     | Shubham    | 将实验迁移到 Markdown 并添加到 GitLab 课程仓库中 |-->


<hr>


## <h3 align="center"> © IBM Corporation。保留所有权利。 <h3/>
